In [2]:
import torch
import torch.nn as nn

layer = nn.Linear(in_features=3, out_features=2, bias=True)

X = torch.randn(4, 3)

# PyTorch forward
Y1 = layer(X)

# Manually implement forward
Y2 = X @ layer.weight.T + layer.bias

print(torch.allclose(Y1, Y2))
# True

print("Input:", X.shape)
print("Weight:", layer.weight.shape)
print("Weight transpose:", layer.weight.T.shape)
print("Bias:", layer.bias.shape)
print("Output:", Y1.shape)

True
Input: torch.Size([4, 3])
Weight: torch.Size([2, 3])
Weight transpose: torch.Size([3, 2])
Bias: torch.Size([2])
Output: torch.Size([4, 2])


In [3]:
loss = Y1.sum()

In [4]:
loss.backward()

In [9]:
layer.weight.shape

torch.Size([2, 3])

In [7]:
layer.bias

Parameter containing:
tensor([-0.3415, -0.1336], requires_grad=True)

In [ ]:
# Layer norm is applied across the features for each sample
# The input and output should have the same shape
import torch
from torch import nn

class RMSLayerNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-5, device=None, dtype=None) -> None:
        super().__init__()
        # This will be broadcast across batch and sequence
        self.weights = nn.Parameter(torch.ones(d_model, device=device, dtype=dtype))
        self.eps = eps
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        in_dtype = x.dtype
        # To avoid overflow
        x = x.to(torch.float32)
        # Compute RMS
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        norm = x * self.weights /rms
        return norm.to(in_dtype)

In [3]:
d_model = 64
eps = 1e-5

norm = RMSLayerNorm(
    d_model=d_model,
    eps=eps,
)

In [1]:
from collections.abc import Callable, Iterable
from typing import Optional
import torch
import math
class SGD(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3):
        if lr < 0:
            raise ValueError(f"Invalid learning rate: {lr}")
        defaults = {"lr": lr}
        super().__init__(params, defaults)

    def step(self, closure: Optional[Callable] = None):
        loss = None if closure is None else closure()
        for group in self.param_groups:
            lr = group["lr"] # Get the learning rate.
            for p in group["params"]:
                if p.grad is None:
                    continue
                state = self.state[p] # Get state associated with p.
                t = state.get("t", 0) # Get iteration number from the state, or 0.
                grad = p.grad.data # Get the gradient of loss with respect to p.
                p.data -= lr / math.sqrt(t + 1) * grad # Update weight tensor in-place.
                state["t"] = t + 1 # Increment iteration number.
        return loss

In [3]:
weights = torch.nn.Parameter(5 * torch.randn((10, 10)))
for lr in [1e1, 1e2, 1e3]:
    print(f"######### lr: {lr} ###########")
    opt = SGD([weights], lr=lr)
    for t in range(10):
        opt.zero_grad() # Reset the gradients for all learnable parameters.
        loss = (weights**2).mean() # Compute a scalar loss value.
        print(loss.cpu().item())
        loss.backward() # Run backward pass, which computes gradients.
        opt.step() # Run optimizer step.

######### lr: 10.0 ###########
22.953838348388672
14.69045639038086
10.829176902770996
8.472674369812012
6.862866401672363
5.690102577209473
4.798847198486328
4.100750923156738
3.5413215160369873
3.0848844051361084
######### lr: 100.0 ###########
2.7070131301879883
2.707012891769409
0.46445003151893616
0.011115334928035736
2.391316285164249e-17
2.6652702839344745e-19
8.974903183851815e-21
5.346411005350995e-22
4.5864972563837594e-23
5.0961081064742274e-24
######### lr: 1000.0 ###########
6.884280158451236e-25
2.4852243665823995e-22
4.2923746153872005e-20
4.774808268914359e-18
3.86759399471711e-16
2.4408945052895384e-14
1.2530754428877189e-12
5.3912669467637286e-11
1.9871064793619553e-09
6.380818717843795e-08


## Compute accounting

1. Write down all the matrix multiplies in a Transformer forward pass.
2. Convert each matrix multiply into FLOPs required.
For this second step, the following fact will be useful:


```Rule: Given 𝐴∈ℝ𝑚×𝑛 and 𝐵∈ℝ𝑛×𝑝, the matrix-matrix product 𝐴𝐵 requires 2𝑚𝑛𝑝 FLOPs.```

### Transformer block operations
1. RMSLayerNorm - no matmul
2. SwiGLU - 3 matmuls
3. Multi head self attention with RoPE - 4 + 2 for attention 


### Transformer block params
1. RMSLayerNorm - d_model params 
2. SwiGLU - 3 d_model * d_ff
3. Multi head self attention with RoPE - 4d_model ** 2

### Transformer LM operations
1. num_layers * transformer_block
2. RMSLayerNorm - no matmul
3. Linear - 1 matmul

### Transformer LM params
1. Embedding - Vocab * d_model
2. num_layers * transformer_block
3. RMSLayerNorm - d_model params
4. Linear - Vocab * d_model


In [1]:
def total_model_trainable_params(
    vocab_size,
    d_model,
    d_ff,
    num_layers
):  
    transformer_block_params = (
        2 * d_model + 3 * (d_model * d_ff) + 4 * (d_model ** 2)
    )
    return 2 * vocab_size * d_model + d_model + num_layers * transformer_block_params

In [2]:
total_model_trainable_params(
    10_000, 512, 1344, 12
)

47608320

In [ ]:
def flops_requirement(
    
)